# BP7 Gate 3 — Decision-Rule-Scheme Benchmark & Champion Selection
**Customer360 Navigator Enterprise Suite — Customer360 Navigator Decision Engine**

## Why this gate looks different from BP1/BP2/BP3's own Gate 3 (and different again from BP4's/BP5's own adaptations)
CRISP-DM's "Modeling" phase maps to Gate 3 for every BP (Section 10.1), but BP7 Gate 1's own real,
live-run `policy.json` (`target_definition`) is explicit that `customer360_priority_decision` is
**NOT a single trained ML target** — it is "a transparent, auditable decision-rule layer (Master
Plan Section 5.1/7) that combines BP1-BP5's own already-computed prediction fields into
`priority_score`, `intervention_flag`, and `recommended_action`", and that the combining
methodology is "a documented, deterministic weighted rule, never a retrained black-box classifier
producing the final score directly." So, mirroring BP4's own aggregation-pipeline-benchmark
adaptation and BP5's own hypothesis-testing/regression/SHAP adaptation, this gate adapts the
generic "Model/Classifier Benchmark & Champion Selection" gate into a **decision-rule-scheme
benchmark**: several real, distinct, honestly-labeled candidate weighting schemes for combining
BP2/BP3/BP4's real Gate 2 fields into `priority_score`/`intervention_flag`/`recommended_action`,
benchmarked on real structural/auditability criteria (never a fabricated accuracy metric — there is
no labeled `customer360_priority_decision` column in the real CFPB extract), with a champion
selected by a real, live-computed criterion — never hardcoded by candidate name.

## The empirical question Gate 1 left open for this gate
Gate 1's own `policy.json` (`assumptions[0]`) named a real, specific concern: **BP2's
`LOW_FRICTION` class and BP3's positive class (`intervention_required=1`) are both defined on the
identical real CFPB value `'Closed with monetary relief'`** — not leakage between BP2 and BP3 (each
already disclosed this at its own Gate 1), but Gate 1 explicitly required this gate to **empirically
check** whether BP2's and BP3's predicted outputs are independent, additively-combinable signals
before finalizing any weights. Section 5 below is that real, live check (chi-square test of
independence + Cramer's V, reusing `models.bp5_driver_association.chi_square_cramers_v`
**unmodified**, HYPER cross-BP reuse — plus the specific literal number Gate 1's assumptions text
named: the real `P(bp3_predicted_label==1 | bp2_predicted_label=='LOW_FRICTION')`, against the real
overall base rate). Its real, live-measured Cramer's V then feeds two of this gate's four
candidates' weight-redundancy discount directly — never a hardcoded correlation figure.

## The 4 candidate rule schemes
Every candidate produces `priority_score` via the identical formula shape — a weighted average of
BP2's real ordinal friction rank, BP3's real intervention probability, and BP4's real cluster
review-priority score, renormalized per row over whichever of the three are actually available
(BP4's real join coverage is ~97.5%, never 100%) — differing ONLY in how each weight is set. BP1
(taxonomy context) and BP5 (outcome-level qualitative association) are **never** a numeric weight,
per Gate 1's own `upstream_input_contract` (`OPTIONAL_CONTEXT_ONLY` for BP1) and the Gate 2 module's
own docstring for BP5 ("never a numeric weight, never a causal claim") — both are recorded into
`reason_codes` as real context only.

- **`equal_weight_baseline`** — BP2/BP3/BP4 weighted 1:1:1, an honestly-labeled naive baseline
  (plays the same "basic/baseline" role `pandas_groupby`/`logistic_regression` play in BP4's/BP1-3's
  own Gate 3 benchmarks).
- **`domain_informed_weighted`** — weighted by each upstream BP's own real, already-computed,
  already real-run-confirmed validated-performance metric: BP2's real Gate 3 champion held-out test
  macro-avg F1 and BP3's real Gate 4 held-out test PR-AUC, both read **live** from each BP's own
  delivered artifact (never hardcoded as a literal in source). BP4 fits no classifier and has no
  comparable held-out predictive metric, so it is weighted by its own real, live-computed Gate 2
  join-coverage fraction instead — a disclosed heuristic, not an invented accuracy number.
- **`correlation_aware`** — identical to `domain_informed_weighted`, except BP2's and BP3's raw
  weights are additionally multiplied by `(1 − cramers_v)`, the real, live-measured association
  strength from Section 5 above — BP4's weight is untouched (it is not part of the correlated
  pair Gate 1 flagged). This directly operationalizes Gate 1's own instruction to "let the finding
  inform ... an approach that doesn't naively double-weight correlated signals."
- **`correlation_aware_plus_lr_diagnostic`** — identical `priority_score`/`intervention_flag`
  formula to `correlation_aware` (same weights, same score, same decision), **plus** the
  interpretable-model variant Gate 1's own `policy.json` explicitly permits: *"A Gate 3/4 scope
  decision MAY evaluate an interpretable model (logistic regression specifically) as one additional
  reason-coded input field into the rule framework — the framework, its weights, its thresholds, and
  the final decision stay the deterministic, auditable layer regardless."* A real, bounded
  `sklearn.LogisticRegression` is fit predicting BP3's own already-computed `bp3_predicted_label`
  from BP2's/BP4's/BP1's OTHER already-computed fields (`bp2_confidence`,
  `bp4_review_priority_score`, a `bp1_context_available` flag) — never a raw/barred CFPB column —
  purely diagnostic: it surfaces which of those fields correlate with BP3's own assessment. Its
  real held-out ROC-AUC/coefficients are saved as one additional disclosed artifact; it is **never**
  substituted for `priority_score`/`intervention_flag`/`recommended_action`, which stay this gate's
  deterministic weighted rule regardless (Gate 1's explicit, non-negotiable constraint).

## The real benchmark criteria — never a fabricated accuracy number
`customer360_priority_decision` has no labeled CFPB column, so this gate never reports a
precision/recall/F1 "accuracy" claim. Instead, every candidate is benchmarked on real, structural,
auditability-relevant properties computed directly from the real Gate 2 Gold layer:
- **Coverage** — % of rows receiving a non-degenerate `priority_score` (never
  `UNSCORED_MISSING_UPSTREAM_INPUT` in practice, since BP2/BP3 both carry real 100% coverage per
  Gate 2's own `gate2_rescoring_summary.json`, but checked live, never assumed).
- **Score-distribution sanity** — no NaN/inf, bounded to `[0, 1]`, non-degenerate spread
  (`std > 0`) — real, computed statistics, not assumed.
- **BP3-coherence** (`bp3_agreement_rate`) — a real, disclosed sanity cross-check between
  `intervention_flag` and BP3's own already-validated `bp3_predicted_label`, explicitly **not**
  presented as an accuracy claim (it is expected to run higher for a candidate that weights BP3 more
  heavily, by construction — disclosed as a known property of this metric, not hidden).
- **Transparency** — every scored row must have non-empty `reason_codes` (asserted structurally,
  100% required, never merely "usually").

## Champion selection — recomputed live, never hardcoded by candidate name
Mirrors BP4 Gate 3's own `CHAMPION = min(timing_results, key=lambda n:
timing_results[n]["min_seconds"])` idiom exactly: among candidates that pass every structural
check above, the champion is the one with the lowest real, live-computed
`redundancy_double_counting_score = cramers_v × (weight_bp2_normalized + weight_bp3_normalized)`
— i.e. the real, computed answer to "which candidate best avoids naively double-weighting the
empirically-correlated BP2/BP3 pair Gate 1 flagged" — with ties broken by higher real
`avg_reason_codes_per_row` (richer disclosure), then higher real `bp3_agreement_rate`, then by
whether that candidate has a real fitted LR diagnostic available (a structural fact read off that
candidate's own benchmark row, never a name comparison).

## The real disparate-impact carry-forward check (or its honest deferral)
BP3's own real Gate 4/5 disparate-impact finding (`adverse_impact_ratio_recomputed = 0.139`,
`flagged: true`, grouped by BP3's own real `tags_group`) is a real, already-closed governance item
Gate 1's own `policy.json` (`compliance_touchpoint.ecoa_reg_b`) commits to re-checking against BP7's
own champion `intervention_flag`. That check needs a Complaint-ID-keyed `tags_group` column on the
real Gate 2 Gold layer. This gate performs a real, live check of whether Gate 2 actually preserved
one — Gate 2's re-scoring join does not carry `tags_group` forward (it lives only in BP3's own
held-out test-split artifact, keyed by a `row_index` local to that split, not `Complaint ID`).
Reconstructing it here would require reading the raw, barred `Tags` column as a BP7 input, which
Gate 1's `leakage_rules` structurally bars. This gate reports that real finding honestly and defers
the check to Gate 4 — never a silent skip, and never a bar relaxation to force it into Gate 3.

## Standing rules this notebook follows
- **Execution boundary**: Claude wrote this notebook; you run it. Every real number below —
  every coverage percentage, Cramer's V, weight, score statistic, and LR coefficient — is a real
  measurement from your own machine's run against your own real Gate 2 Gold layer, never simulated.
- **Zero-fabrication**: no financial-impact, illustrative, or assumption-based content anywhere;
  every weight traces to a real, already-computed upstream metric or a stated, disclosed heuristic.
- **WARP**: `configure_performance()` first, before any heavy import, same as every other notebook.
- **HYPER**: reuses `src/utils/bp1_config_sync.py` (sixth BP to do so) and
  `models.bp5_driver_association.chi_square_cramers_v` **unmodified** (true cross-BP reuse); every
  new Gate 3 function lives in `src/features/bp7_decision_engine_features.py` (extended, not
  duplicated — Gate 4+ will import these same functions).
- **Idempotent**: re-running this notebook overwrites this gate's own config block and artifacts in
  place; every other gate's block is preserved verbatim regardless of position.
- **PROJECT_STRUCTURE_LOCKED.md rule #3**: same project-root resolver as every other notebook.
- **No test coverage added here** — matching every other BP's own precedent (BP2-BP5 all added
  their first pytest coverage at their own Gate 6, confirmed in BP7 Gate 2's own delivery report),
  BP7's own first tests are deferred to its own Gate 6.

## Outputs (idempotent overwrite-in-place)
- `notebooks/bp7_customer_navigator_decision_engine/artifacts/gate3_benchmark_results.csv` — one
  row per candidate (matches BP4 Gate 3's own artifact-naming convention).
- `notebooks/bp7_customer_navigator_decision_engine/artifacts/gate3_bp2_bp3_correlation_check.json`
- `notebooks/bp7_customer_navigator_decision_engine/artifacts/gate3_lr_diagnostic.json`
- `notebooks/bp7_customer_navigator_decision_engine/artifacts/gate3_disparate_impact_carry_forward_check.json`
- `notebooks/bp7_customer_navigator_decision_engine/artifacts/gate3_decision_rule_benchmark_summary.json`
- `configs/bp7_customer_navigator_decision_engine.yaml` — Gate 3 marker block appended/overwritten.

## Prerequisites
BP7 Gate 2 must have been real-run at least once — this notebook checks the Gate 2 config block
and `gate2_rescoring_summary.json` live and raises a clear error if either is missing. It also
reads BP2's real `gate3_champion_test_classification_report.json` and BP3's real
`gate4_statistical_validation.json` live (both already real-run-confirmed prerequisites of BP7
Gate 1/2) — never hardcoding either BP's own metric.

## If a structural check below fails
It raises `AssertionError` naming the failing check. A check failing because **zero** candidates
pass every structural check is a genuine data-integrity problem and must never be worked around by
loosening a check. This gate never reconstructs `tags_group` from the raw `Tags` column to force
the disparate-impact check into this gate — that check is honestly deferred to Gate 4 instead.


In [ ]:
"""
Customer360 Navigator Enterprise Suite - BP7 Gate 3 decision-rule-scheme benchmark notebook.
Single consolidated code cell (platform convention). Idempotent - safe to re-run.
"""

import os, sys, json, warnings
from datetime import datetime, timezone
from pathlib import Path

warnings.filterwarnings("ignore")


# ============================================================
# SECTION 1: Project root resolution (PROJECT_STRUCTURE_LOCKED.md rule #3)
# ============================================================
def _find_project_root() -> Path:
    marker = "PROJECT_STRUCTURE_LOCKED.md"
    env_override = os.environ.get("C360_PROJECT_ROOT")
    if env_override:
        if (Path(env_override) / marker).exists():
            return Path(env_override)
        raise RuntimeError(
            f"C360_PROJECT_ROOT is set to {env_override!r} but {marker} was not found there. "
            "Fix the environment variable rather than removing this check."
        )

    start = Path.cwd()
    cur = start
    for _ in range(8):
        if (cur / marker).exists():
            return cur
        if cur.parent == cur:
            break
        cur = cur.parent

    for depth_root, dirnames, filenames in os.walk(start):
        rel_depth = len(Path(depth_root).relative_to(start).parts)
        if rel_depth > 3:
            dirnames[:] = []
            continue
        dirnames[:] = [d for d in dirnames if not d.startswith(".")]
        if marker in filenames:
            return Path(depth_root)

    raise RuntimeError(
        f"Could not resolve PROJECT_ROOT: no {marker} found by walking up from {start}, nor by "
        "searching up to 3 levels below it. Fix: add a cell at the TOP of this notebook (before "
        "this cell runs) with:\n"
        '    import os; os.environ["C360_PROJECT_ROOT"] = r"C:\\Users\\rnand\\Documents\\'
        'Customer360_Navigator_Enterprise_Suite"\n'
        "then re-run from the top."
    )


PROJECT_ROOT = _find_project_root()
sys.path.insert(0, str(PROJECT_ROOT / "src"))
print(f"[OK] Project root resolved: {PROJECT_ROOT.name}")

# ============================================================
# SECTION 2: WARP performance configuration - FIRST, before any heavy import
# ============================================================
from utils.performance_setup import configure_performance, memory_headroom_gb  # noqa: E402

perf_summary = configure_performance(project_root=PROJECT_ROOT, verbose=True)
print(f"[WARP] Headroom before heavy work: {memory_headroom_gb()} GB")

# ============================================================
# SECTION 3: Heavy imports (only after WARP configuration)
# ============================================================
import pandas as pd  # noqa: E402
import polars as pl  # noqa: E402
import yaml  # noqa: E402
from IPython.display import display  # noqa: E402

from utils.bp1_config_sync import write_gate_block  # noqa: E402
from features.bp7_decision_engine_features import (  # noqa: E402
    CANDIDATE_NAMES,
    DEFAULT_INTERVENTION_THRESHOLD,
    load_bp2_friction_ordinal_ranks,
    load_upstream_validated_metrics,
    compute_bp4_join_coverage,
    compute_bp2_bp3_correlation,
    attach_normalized_signal_columns,
    compute_candidate_raw_weights,
    normalize_candidate_weights,
    score_priority_rule,
    benchmark_candidate,
    fit_lr_diagnostic,
    check_disparate_impact_carry_forward,
    select_champion,
)

CONFIGS_DIR = PROJECT_ROOT / "configs"
DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"
ARTIFACTS_DIR = PROJECT_ROOT / "notebooks" / "bp7_customer_navigator_decision_engine" / "artifacts"
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)

BP7_CONFIG_PATH = CONFIGS_DIR / "bp7_customer_navigator_decision_engine.yaml"
GOLD_PATH = DATA_PROCESSED / "cfpb_decision_engine_context_gold.parquet"
GATE2_SUMMARY_PATH = ARTIFACTS_DIR / "gate2_rescoring_summary.json"

CORRELATION_CHECK_PATH = ARTIFACTS_DIR / "gate3_bp2_bp3_correlation_check.json"
LR_DIAGNOSTIC_PATH = ARTIFACTS_DIR / "gate3_lr_diagnostic.json"
DISPARATE_IMPACT_DEFERRAL_PATH = ARTIFACTS_DIR / "gate3_disparate_impact_carry_forward_check.json"
BENCHMARK_CSV_PATH = ARTIFACTS_DIR / "gate3_benchmark_results.csv"
SUMMARY_JSON_PATH = ARTIFACTS_DIR / "gate3_decision_rule_benchmark_summary.json"

for p in (BP7_CONFIG_PATH, GOLD_PATH, GATE2_SUMMARY_PATH):
    if not p.exists():
        raise FileNotFoundError(
            f"Required input not found: {p}. Confirm BP7 Gate 2 has been real-run at least once "
            "(the Gold-layer parquet and its own summary artifact are both Gate 2's own real "
            "output)."
        )

# ============================================================
# SECTION 4: Gate 2 prerequisite check (live) - never trusted from memory, re-read every run.
# ============================================================
with open(BP7_CONFIG_PATH, "r", encoding="utf-8") as f:
    full_config_text = f.read()
full_config = yaml.safe_load(full_config_text)
gate1_status_confirmed = "gate1_confirmed" in str(full_config.get("status", ""))
gate2_block_marker = "# --- Gate 2 (Data Verification & Feature/Taxonomy Engineering) results"
gate2_block_present = gate2_block_marker in full_config_text
gate2_confirmed = (
    gate1_status_confirmed
    and gate2_block_present
    and full_config.get("bp2_bundle_available") is True
    and full_config.get("bp3_bundle_available") is True
)
assert gate2_confirmed, (
    "BP7 Gate 2 does not appear to have completed successfully (gate1_status_confirmed="
    f"{gate1_status_confirmed}, gate2_block_present={gate2_block_present}, "
    f"bp2_bundle_available={full_config.get('bp2_bundle_available')!r}, "
    f"bp3_bundle_available={full_config.get('bp3_bundle_available')!r}). Run Gate 2 for real "
    "before Gate 3."
)
print(
    f"[OK] Gate 2 prerequisite confirmed (gold_layer_rows_written="
    f"{full_config.get('gold_layer_rows_written'):,})."
)

with open(GATE2_SUMMARY_PATH, "r", encoding="utf-8") as f:
    gate2_summary = json.load(f)

gold_pl = pl.read_parquet(GOLD_PATH)
live_row_count = gold_pl.height
row_count_matches_gate2 = live_row_count == gate2_summary.get("gold_layer_rows_written")
print(
    f"[OK] Real Gold layer loaded live: {live_row_count:,} rows x {gold_pl.width} cols "
    f"(matches Gate 2's own recorded gold_layer_rows_written: {row_count_matches_gate2})."
)
assert row_count_matches_gate2, (
    "Real Gold layer row count no longer matches BP7 Gate 2's own recorded "
    f"gold_layer_rows_written ({gate2_summary.get('gold_layer_rows_written')}) - re-run Gate 2 "
    "before trusting Gate 3's own results below."
)

# ============================================================
# SECTION 5: The real, live empirical check Gate 1's own policy.json (assumptions) required -
# is BP2's friction-tier prediction independent of BP3's intervention-probability prediction?
# ============================================================
correlation_result = compute_bp2_bp3_correlation(gold_pl)
BP2_BP3_CRAMERS_V = correlation_result["chi_square_cramers_v"]["cramers_v"]
print(
    f"\n[FINDING] BP2 vs BP3 real association: cramers_v={BP2_BP3_CRAMERS_V:.4f} "
    f"({correlation_result['chi_square_cramers_v']['association_strength']}), "
    f"p_value={correlation_result['chi_square_cramers_v']['p_value']:.4g}"
)
print(
    f"[FINDING] Real P(bp3_predicted_label==1 | bp2_predicted_label=='LOW_FRICTION') = "
    f"{correlation_result['low_friction_bp3_positive_rate']} vs real overall base rate = "
    f"{correlation_result['overall_bp3_positive_rate']} (n_low_friction="
    f"{correlation_result['n_low_friction_rows']:,})."
)

with open(CORRELATION_CHECK_PATH, "w", encoding="utf-8") as f:
    json.dump(correlation_result, f, indent=2, default=str)
print(f"[SAVED] {CORRELATION_CHECK_PATH.relative_to(PROJECT_ROOT)}")

# ============================================================
# SECTION 6: Each upstream BP's own real, already-computed validated-performance/coverage metric
# - read LIVE from each BP's own delivered artifact, never hardcoded.
# ============================================================
upstream_metrics = load_upstream_validated_metrics(PROJECT_ROOT)
bp4_join_coverage = compute_bp4_join_coverage(gold_pl)
print(
    f"\n[OK] Real upstream validated metrics (live): bp2_f1_macro="
    f"{upstream_metrics['bp2_f1_macro']:.4f} (source: {upstream_metrics['bp2_f1_macro_source']}), "
    f"bp3_pr_auc={upstream_metrics['bp3_pr_auc']:.4f} (source: "
    f"{upstream_metrics['bp3_pr_auc_source']})"
)
print(f"[OK] Real BP4 join coverage (live) = {bp4_join_coverage:.4%}")

# ============================================================
# SECTION 7: BP2's own real, already-governance-reviewed friction-severity ordinal taxonomy,
# read live - never hardcoded/assumed. Attach the 3 real, bounded [0,1] normalized signal columns
# the weighted rule below combines.
# ============================================================
friction_ordinal_ranks = load_bp2_friction_ordinal_ranks(PROJECT_ROOT)
print(f"\n[OK] BP2 real friction ordinal ranks (live from taxonomy config): {friction_ordinal_ranks}")
signal_lazy = attach_normalized_signal_columns(gold_pl.lazy(), friction_ordinal_ranks)

# ============================================================
# SECTION 8: Raw + normalized weights for every candidate rule scheme - every number traces to a
# real, already-computed upstream metric (Section 6) or a real, live-computed association
# statistic (Section 5), never invented.
# ============================================================
raw_weights_by_candidate = compute_candidate_raw_weights(
    upstream_metrics, bp4_join_coverage, BP2_BP3_CRAMERS_V
)
normalized_weights_by_candidate = {
    name: normalize_candidate_weights(w) for name, w in raw_weights_by_candidate.items()
}
print("\n=== CANDIDATE RULE SCHEMES - REAL, LIVE-COMPUTED WEIGHTS ===")
for name in CANDIDATE_NAMES:
    print(f"  {name}:")
    print(f"    raw        = {raw_weights_by_candidate[name]}")
    print(f"    normalized = {normalized_weights_by_candidate[name]}")

# ============================================================
# SECTION 9: The optional interpretable-model reason-coded input Gate 1's policy.json explicitly
# permits - fit ONLY on already-available real upstream PREDICTED fields, diagnostic only.
# ============================================================
lr_diagnostic = fit_lr_diagnostic(
    gold_pl, sample_size=200_000, random_state=full_config.get("random_state", 42)
)
print(
    f"\n[OK] LR diagnostic (real, bounded sample n={lr_diagnostic['n_rows_fit_sample']:,}): "
    f"held_out_roc_auc={lr_diagnostic['held_out_roc_auc']:.4f}, "
    f"held_out_accuracy={lr_diagnostic['held_out_accuracy']:.4f}"
)
print(f"[OK] LR diagnostic real coefficients: {lr_diagnostic['coefficients']}")
with open(LR_DIAGNOSTIC_PATH, "w", encoding="utf-8") as f:
    json.dump(lr_diagnostic, f, indent=2, default=str)
print(f"[SAVED] {LR_DIAGNOSTIC_PATH.relative_to(PROJECT_ROOT)}")

# ============================================================
# SECTION 10: Score every candidate on the real, full Gold-layer population and compute its real
# benchmark metrics (never a fabricated accuracy - there is no labeled ground-truth column).
# ============================================================
benchmark_rows: dict[str, dict] = {}
for name in CANDIDATE_NAMES:
    scored_pl = score_priority_rule(
        signal_lazy, normalized_weights_by_candidate[name], DEFAULT_INTERVENTION_THRESHOLD
    ).collect()
    row = benchmark_candidate(
        scored_pl, name, normalized_weights_by_candidate[name], raw_weights_by_candidate[name]
    )
    if name == "correlation_aware_plus_lr_diagnostic":
        row["lr_diagnostic_held_out_roc_auc"] = lr_diagnostic["held_out_roc_auc"]
        row["lr_diagnostic_n_rows_fit_sample"] = lr_diagnostic["n_rows_fit_sample"]
    benchmark_rows[name] = row
    print(
        f"\n[BENCHMARK] {name}: coverage={row['coverage_pct']}% "
        f"score_std={row['score_std']:.4f} bounded_0_1={row['score_bounded_0_1']} "
        f"non_degenerate={row['score_non_degenerate']} "
        f"reason_codes_all_nonempty={row['reason_codes_all_nonempty']} "
        f"avg_reason_codes={row['avg_reason_codes_per_row']} "
        f"intervention_flag_rate={row['intervention_flag_rate']} "
        f"bp3_agreement_rate={row['bp3_agreement_rate']} "
        f"structurally_passes={row['structurally_passes']}"
    )

benchmark_df = pd.DataFrame([benchmark_rows[name] for name in CANDIDATE_NAMES])
display(benchmark_df)

# ============================================================
# SECTION 11: Champion selection - recomputed live from the real benchmark rows above, never
# hardcoded by candidate name (this project's own established idiom, see BP4 Gate 3's own
# min(timing_results, key=...) selection).
# ============================================================
CHAMPION = select_champion(benchmark_rows, BP2_BP3_CRAMERS_V)
champion_row = benchmark_rows[CHAMPION]
champion_redundancy_score = BP2_BP3_CRAMERS_V * (
    champion_row["weight_bp2_normalized"] + champion_row["weight_bp3_normalized"]
)
print(f"\n[CHAMPION] {CHAMPION}")
print(
    f"[CHAMPION] weights: bp2={champion_row['weight_bp2_normalized']:.4f}, "
    f"bp3={champion_row['weight_bp3_normalized']:.4f}, bp4={champion_row['weight_bp4_normalized']:.4f}"
)
print(f"[CHAMPION] redundancy_double_counting_score={champion_redundancy_score:.6f}")
print("\n=== REDUNDANCY-DOUBLE-COUNTING SCORE PER CANDIDATE (real, live-computed) ===")
for name in CANDIDATE_NAMES:
    row = benchmark_rows[name]
    redundancy = BP2_BP3_CRAMERS_V * (row["weight_bp2_normalized"] + row["weight_bp3_normalized"])
    print(f"  {name}: {redundancy:.6f}{'  <-- CHAMPION' if name == CHAMPION else ''}")

# ============================================================
# SECTION 12: Disparate-impact carry-forward check (BP7 Gate 1's own compliance_touchpoint.
# ecoa_reg_b commitment) - real, live check of whether it can run at Gate 3, or must be honestly
# deferred to Gate 4. Never reconstructed here from the raw, barred 'Tags' column.
# ============================================================
disparate_impact_check = check_disparate_impact_carry_forward(gold_pl.columns)
print(
    f"\n[DISPARATE IMPACT CARRY-FORWARD] tags_group present in Gate 2 Gold layer: "
    f"{disparate_impact_check['tags_group_column_present_in_gate2_gold_layer']}"
)
if disparate_impact_check["deferred_to_gate4"]:
    print(f"[DEFERRED TO GATE 4] {disparate_impact_check['deferral_reason']}")
with open(DISPARATE_IMPACT_DEFERRAL_PATH, "w", encoding="utf-8") as f:
    json.dump(disparate_impact_check, f, indent=2, default=str)
print(f"[SAVED] {DISPARATE_IMPACT_DEFERRAL_PATH.relative_to(PROJECT_ROOT)}")

# ============================================================
# SECTION 13: Write the real per-candidate benchmark results table (every candidate, including
# any that failed a structural check - never only the winner).
# ============================================================
benchmark_df.to_csv(BENCHMARK_CSV_PATH, index=False)
print(f"\n[SAVED] {BENCHMARK_CSV_PATH.relative_to(PROJECT_ROOT)}")

# ============================================================
# SECTION 14: Write the Gate 3 master summary JSON artifact.
# ============================================================
gate3_summary = {
    "bp_id": "bp7",
    "gate": 3,
    "generated_at_utc": datetime.now(timezone.utc).isoformat(),
    "live_row_count": live_row_count,
    "gate2_recorded_row_count": gate2_summary.get("gold_layer_rows_written"),
    "candidate_names": CANDIDATE_NAMES,
    "intervention_threshold": DEFAULT_INTERVENTION_THRESHOLD,
    "bp2_bp3_correlation_check": {
        "cramers_v": BP2_BP3_CRAMERS_V,
        "association_strength": correlation_result["chi_square_cramers_v"]["association_strength"],
        "low_friction_bp3_positive_rate": correlation_result["low_friction_bp3_positive_rate"],
        "overall_bp3_positive_rate": correlation_result["overall_bp3_positive_rate"],
        "path": str(CORRELATION_CHECK_PATH.relative_to(PROJECT_ROOT).as_posix()),
    },
    "upstream_validated_metrics": upstream_metrics,
    "bp4_join_coverage": bp4_join_coverage,
    "weights_by_candidate": {
        name: {
            "raw": raw_weights_by_candidate[name],
            "normalized": normalized_weights_by_candidate[name],
        }
        for name in CANDIDATE_NAMES
    },
    "lr_diagnostic": {
        "held_out_roc_auc": lr_diagnostic["held_out_roc_auc"],
        "held_out_accuracy": lr_diagnostic["held_out_accuracy"],
        "n_rows_fit_sample": lr_diagnostic["n_rows_fit_sample"],
        "path": str(LR_DIAGNOSTIC_PATH.relative_to(PROJECT_ROOT).as_posix()),
    },
    "champion_rule_scheme": CHAMPION,
    "champion_redundancy_double_counting_score": champion_redundancy_score,
    "champion_weights_normalized": {
        "bp2": champion_row["weight_bp2_normalized"],
        "bp3": champion_row["weight_bp3_normalized"],
        "bp4": champion_row["weight_bp4_normalized"],
    },
    "champion_bp3_agreement_rate": champion_row["bp3_agreement_rate"],
    "champion_coverage_pct": champion_row["coverage_pct"],
    "disparate_impact_carry_forward_check": {
        "deferred_to_gate4": disparate_impact_check["deferred_to_gate4"],
        "deferral_reason": disparate_impact_check["deferral_reason"],
        "path": str(DISPARATE_IMPACT_DEFERRAL_PATH.relative_to(PROJECT_ROOT).as_posix()),
    },
    "benchmark_results_path": str(BENCHMARK_CSV_PATH.relative_to(PROJECT_ROOT).as_posix()),
}
with open(SUMMARY_JSON_PATH, "w", encoding="utf-8") as f:
    json.dump(gate3_summary, f, indent=2, default=str)
print(f"[SAVED] {SUMMARY_JSON_PATH.relative_to(PROJECT_ROOT)}")

# ============================================================
# SECTION 15: Write the Gate 3 config block (marker-based, order-independent - reuses
# bp1_config_sync.py unmodified, sixth BP to do so).
# ============================================================
gate3_marker = (
    "# --- Gate 3 (Decision-Rule-Scheme Benchmark & Champion Selection) results "
    "(appended, idempotent overwrite) ---"
)
n_structurally_passing_for_block = sum(1 for r in benchmark_rows.values() if r["structurally_passes"])
bp2_bp3_association_strength = correlation_result["chi_square_cramers_v"]["association_strength"]
bp2_bp3_low_friction_conditional_positive_rate = correlation_result["low_friction_bp3_positive_rate"]
gate3_block_lines = (
    [
        f"candidates_total: {len(CANDIDATE_NAMES)}",
        f"candidates_structurally_passing: {n_structurally_passing_for_block}",
        "candidate_structurally_passes:",
    ]
    + [f"  {name}: {benchmark_rows[name]['structurally_passes']}" for name in CANDIDATE_NAMES]
    + [
        f"bp2_bp3_cramers_v: {BP2_BP3_CRAMERS_V}",
        f'bp2_bp3_association_strength: "{bp2_bp3_association_strength}"',
        "bp2_bp3_low_friction_conditional_positive_rate: "
        f"{bp2_bp3_low_friction_conditional_positive_rate}",
        f"bp4_join_coverage: {bp4_join_coverage}",
        f"bp2_f1_macro: {upstream_metrics['bp2_f1_macro']}",
        f"bp3_pr_auc: {upstream_metrics['bp3_pr_auc']}",
        f"intervention_threshold: {DEFAULT_INTERVENTION_THRESHOLD}",
        f'champion_rule_scheme: "{CHAMPION}"',
        f"champion_weight_bp2: {champion_row['weight_bp2_normalized']}",
        f"champion_weight_bp3: {champion_row['weight_bp3_normalized']}",
        f"champion_weight_bp4: {champion_row['weight_bp4_normalized']}",
        f"champion_redundancy_double_counting_score: {champion_redundancy_score}",
        f"champion_bp3_agreement_rate: {champion_row['bp3_agreement_rate']}",
        f"champion_coverage_pct: {champion_row['coverage_pct']}",
        f"lr_diagnostic_held_out_roc_auc: {lr_diagnostic['held_out_roc_auc']}",
        f"disparate_impact_carry_forward_deferred_to_gate4: {disparate_impact_check['deferred_to_gate4']}",
        f'bp2_bp3_correlation_check_path: "{CORRELATION_CHECK_PATH.relative_to(PROJECT_ROOT).as_posix()}"',
        f'lr_diagnostic_path: "{LR_DIAGNOSTIC_PATH.relative_to(PROJECT_ROOT).as_posix()}"',
        "disparate_impact_carry_forward_check_path: "
        f'"{DISPARATE_IMPACT_DEFERRAL_PATH.relative_to(PROJECT_ROOT).as_posix()}"',
        f'benchmark_results_path: "{BENCHMARK_CSV_PATH.relative_to(PROJECT_ROOT).as_posix()}"',
        f'gate3_summary_path: "{SUMMARY_JSON_PATH.relative_to(PROJECT_ROOT).as_posix()}"',
    ]
)
write_gate_block(BP7_CONFIG_PATH, gate3_marker, gate3_block_lines)
print(f"[SAVED] gate3 block written to {BP7_CONFIG_PATH.relative_to(PROJECT_ROOT)}")

with open(BP7_CONFIG_PATH, "r", encoding="utf-8") as f:
    _post_write_config_text = f.read()
gate3_block_actually_written = gate3_marker in _post_write_config_text

# ============================================================
# SECTION 16: Structural integrity checks - raise AssertionError, never silently pass.
# ============================================================
checks = {
    "gate2_prerequisite_confirmed": gate2_confirmed,
    "gold_layer_row_count_matches_gate2_recorded": row_count_matches_gate2,
    "bp2_bp3_cramers_v_in_valid_range": 0.0 <= BP2_BP3_CRAMERS_V <= 1.0,
    "bp4_join_coverage_in_valid_range": 0.0 < bp4_join_coverage <= 1.0,
    "every_candidate_weights_sum_to_one": all(
        abs(sum(normalize_candidate_weights(raw_weights_by_candidate[name]).values()) - 1.0) < 1e-9
        for name in CANDIDATE_NAMES
    ),
    "correlation_aware_pair_weights_le_domain_informed_pair_weights": (
        raw_weights_by_candidate["correlation_aware"]["bp2"]
        <= raw_weights_by_candidate["domain_informed_weighted"]["bp2"]
        and raw_weights_by_candidate["correlation_aware"]["bp3"]
        <= raw_weights_by_candidate["domain_informed_weighted"]["bp3"]
    ),
    "at_least_one_candidate_structurally_passes": any(
        r["structurally_passes"] for r in benchmark_rows.values()
    ),
    "champion_is_among_structurally_passing_candidates": benchmark_rows[CHAMPION]["structurally_passes"],
    "champion_has_lowest_redundancy_score_among_passing": champion_redundancy_score
    == min(
        BP2_BP3_CRAMERS_V * (r["weight_bp2_normalized"] + r["weight_bp3_normalized"])
        for r in benchmark_rows.values()
        if r["structurally_passes"]
    ),
    "every_candidate_reason_codes_all_nonempty": all(
        r["reason_codes_all_nonempty"] for r in benchmark_rows.values()
    ),
    "lr_diagnostic_roc_auc_in_valid_range": 0.0 <= lr_diagnostic["held_out_roc_auc"] <= 1.0,
    "disparate_impact_check_disclosed_never_silently_skipped": (
        disparate_impact_check["disparate_impact_carry_forward_check_performed_at_gate3"]
        or disparate_impact_check["deferred_to_gate4"]
    ),
    "correlation_check_json_written": CORRELATION_CHECK_PATH.exists(),
    "lr_diagnostic_json_written": LR_DIAGNOSTIC_PATH.exists(),
    "disparate_impact_deferral_json_written": DISPARATE_IMPACT_DEFERRAL_PATH.exists(),
    "benchmark_csv_written": BENCHMARK_CSV_PATH.exists(),
    "benchmark_csv_row_count_matches_candidate_count": len(benchmark_df) == len(CANDIDATE_NAMES),
    "gate3_summary_json_written": SUMMARY_JSON_PATH.exists(),
    "config_gate3_block_written": gate3_block_actually_written,
}

print("\n=== INTEGRITY CHECKS ===")
for check_name, passed in checks.items():
    result_label = "[PASS]" if passed else "[FAIL]"
    print(f"{result_label} {check_name}")
    assert passed, f"[CHECK FAILED] {check_name}"

n_structurally_passing = sum(1 for r in benchmark_rows.values() if r["structurally_passes"])
disparate_impact_disposition = (
    "performed at Gate 3"
    if not disparate_impact_check["deferred_to_gate4"]
    else "honestly deferred to Gate 4"
)
print(
    f"\n[ALL CHECKS PASSED] BP7 Gate 3 complete - {n_structurally_passing}/{len(CANDIDATE_NAMES)} "
    f"candidates structurally passed, champion='{CHAMPION}' (real bp2_bp3_cramers_v="
    f"{BP2_BP3_CRAMERS_V:.4f}, redundancy_double_counting_score={champion_redundancy_score:.6f}). "
    f"Disparate-impact carry-forward check: {disparate_impact_disposition}. Proceed to BP7 Gate 4 next."
)
